# Ingeniería de features

El EDA dejó dicho qué patrones distinguen el fraude y cuánto vale cada uno. Quedan dos preguntas:

1. **¿Cómo se traduce cada hallazgo a una columna** que el modelo pueda usar?
2. **¿Esa traducción usa información que en producción no vamos a tener?** Codificar `j` por su tasa de
   fraude obliga a calcular esa tasa con alguna porción de los datos. Si es la porción equivocada, el
   modelo ve de rebote la respuesta que tiene que predecir.

No se eligen modelos, eso es `03_modeling.ipynb`: los chequeos de la sección 2 entrenan una logística solo
como control. La implementación vive en `fraud_detection.features`.

## 1. Qué recibe el modelo

`build_preprocessor` arma el preprocesador con dos parámetros: `scale`, para usar las mismas features en
la regresión logística y en los árboles, y `use_score`, para medir el modelo sin esa columna si el dueño
del dato confirma que no está disponible al decidir el pago.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from fraud_detection.constants import DATE, N_SPLITS, RANDOM_STATE, TARGET
from fraud_detection.dataset import load_periods
from fraud_detection.features import build_preprocessor
from fraud_detection.modeling import temporal_folds

development, _ = load_periods()
X, y = development.drop(columns=[TARGET]), development[TARGET]
folds = list(temporal_folds(development[DATE], n_splits=N_SPLITS))
train_pos, valid_pos = folds[-1]

preprocesador = build_preprocessor(scale=True).fit(X.iloc[train_pos], y.iloc[train_pos])
columnas = list(preprocesador.get_feature_names_out())
sin_score = build_preprocessor(use_score=False).fit(X.iloc[train_pos], y.iloc[train_pos])

print(f"{len(development):,} filas de desarrollo | {len(folds)} folds temporales")
print(f"{len(columnas)} features, o {len(sin_score.get_feature_names_out())} sin score")
print("\n".join(f"  {columnas[i:i + 6]}" for i in range(0, len(columnas), 6)))

apariciones = development.j.value_counts()
print(
    f"\nj: {development.j.nunique():,} categorías, de las cuales "
    f"{apariciones.eq(1).sum():,} aparecen una sola vez"
)
print(
    f"   de esas, {development.loc[development.j.isin(apariciones[apariciones.eq(1)].index), TARGET].sum():,} "
    f"son fraude"
)

114,506 filas de desarrollo | 4 folds temporales
33 features, o 32 sin score
  ['b', 'c', 'd', 'e', 'f', 'h']
  ['k', 'l', 'm', 'monto', 'score', 'missingindicator_b']
  ['missingindicator_c', 'missingindicator_d', 'missingindicator_f', 'missingindicator_l', 'missingindicator_m', 'a_1']
  ['a_2', 'a_3', 'a_4', 'o_Ausente', 'o_N', 'o_Y']
  ['p_N', 'p_Y', 'g', 'j', 'freq_j', 'n']
  ['hora_sin', 'hora_cos', 'es_madrugada']

j: 7,770 categorías, de las cuales 2,302 aparecen una sola vez
   de esas, 74 son fraude


**Lectura: cada columna sale de un hallazgo del EDA.**

| Hallazgo del EDA | Columna que produce |
|---|---|
| La ausencia de `o` informa, y es el patrón que más rinde | `o_Ausente` como categoría propia, más un `missingindicator_` por cada numérica con nulos |
| `a` es nominal: su tasa no sigue el orden 1-4 | `a_1` a `a_4` en one-hot, nunca como valor numérico |
| `j` tiene miles de valores y su frecuencia tiene un riesgo propio | `j` por target encoding **y** `freq_j` por frecuencia: miden cosas distintas |
| La madrugada tiene más fraude | `hora_sin`, `hora_cos` y `es_madrugada`; la hora es cíclica, así que 23 y 0 tienen que quedar cerca |
| Día de semana y finde no separan | Ninguna: no se construyen features de calendario |
| `k` es ruido | `k` se conserva, como vara para leer importancias |

Las 33 columnas salen de 11 numéricas más 6 indicadores de nulo, 9 one-hot, 2 target-encoded, 1 de
frecuencia, 1 binaria y 3 temporales. `monto` entra como feature, pero el pipeline nunca lo modifica en el
frame original, porque `expected_gain` lo necesita en sus unidades. Con `scale` se estandarizan las
numéricas, el target encoding y la frecuencia, nunca las one-hot, que ya están en 0/1.

## 2. El riesgo: una fuga que no avisa

El target encoding reemplaza cada categoría por su tasa de fraude. Si esa tasa se calculara sobre todo
desarrollo, cada fila de validación habría aportado **su propia etiqueta** al número que el modelo después
lee. La celda anterior muestra el caso extremo: hay **2.302 categorías de `j` que aparecen una sola vez**,
y **74 de ellas son fraude**. Calculada así, la tasa de esas 74 sería 1,0: la feature diría "esto es
fraude".

Un modelo así acierta casi perfecto en validación y no sirve en producción, donde las transacciones
futuras todavía no aportaron su etiqueta a ninguna tabla. Y **no hay error ni warning**: el notebook corre
igual y las métricas suben.

Cuatro chequeos, del más fuerte al más básico, para ver si el pipeline se salva de eso.

### Chequeo 1: etiquetas barajadas

Se baraja `y` y se corre el pipeline completo. Con etiquetas aleatorias no queda nada que aprender, así
que **esperamos AUC 0,5**, el de ordenar al azar (EDA §5). Si el encoding filtrara, habría memorizado
etiquetas que el modelo después reconoce, y daría más.

El mismo pipeline corre con las etiquetas reales, y **es el control**: un pipeline roto también daría 0,5
con etiquetas barajadas. Hace falta ver que aprende cuando hay algo que aprender.

In [2]:
def auc_por_fold(y_usado: pd.Series) -> list[float]:
    """AUC en validación para cada fold temporal, con el pipeline completo."""
    puntajes = []
    for train_idx, valid_idx in folds:
        pipeline = Pipeline(
            [
                ("preprocesamiento", build_preprocessor(scale=True)),
                ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced")),
            ]
        )
        pipeline.fit(X.iloc[train_idx], y_usado.iloc[train_idx])
        puntajes.append(
            roc_auc_score(y_usado.iloc[valid_idx], pipeline.predict_proba(X.iloc[valid_idx])[:, 1])
        )
    return puntajes


y_barajada = pd.Series(
    np.random.RandomState(RANDOM_STATE).permutation(y.to_numpy()), index=y.index, name=TARGET
)
resultados = pd.DataFrame(
    {"etiquetas barajadas": auc_por_fold(y_barajada), "etiquetas reales": auc_por_fold(y)},
    index=[f"fold {i}" for i in range(len(folds))],
)
resultados.loc["media"] = resultados.mean()
display(resultados.round(4))

auc_barajada = resultados.loc["media", "etiquetas barajadas"]
assert (
    0.48 <= auc_barajada <= 0.52
), f"Posible fuga: AUC {auc_barajada:.4f} con etiquetas aleatorias."

,etiquetas barajadas,etiquetas reales
fold 0,0.5065,0.8495
fold 1,0.5010,0.8499
fold 2,0.5028,0.8436
fold 3,0.4917,0.8436
media,0.5005,0.8467


**El pipeline pasa.** Con etiquetas barajadas da 0,5005, indistinguible del azar en los cuatro folds. Si
el encoding hubiera filtrado, esas 74 categorías de una sola aparición con etiqueta de fraude habrían
levantado el número de forma visible.

Y con las etiquetas reales llega a **0,8467**, así que el 0,5 anterior no es un pipeline roto: es un
pipeline que funciona y al que no le queda nada que aprender cuando la etiqueta es ruido.

### Chequeos 2, 3 y 4: determinismo, orden temporal y cobertura

Que `transform` no dependa de las etiquetas de validación, que ningún fold entrene con datos posteriores
a su validación, y que las categorías de `j` que aparecen por primera vez en validación no rompan la
transformación ni generen nulos.

In [3]:
np.testing.assert_allclose(
    preprocesador.transform(X.iloc[valid_pos]), preprocesador.transform(X.iloc[valid_pos])
)

diagnostico = []
for numero, (train_idx, valid_idx) in enumerate(folds):
    transformador = build_preprocessor(scale=True)
    matriz_train = transformador.fit_transform(X.iloc[train_idx], y.iloc[train_idx])
    matriz_valid = transformador.transform(X.iloc[valid_idx])

    fin_train = development[DATE].iloc[train_idx].max()
    inicio_valid = development[DATE].iloc[valid_idx].min()
    assert fin_train < inicio_valid, f"Fold {numero} entrena con datos posteriores a su validación."
    assert not np.isnan(matriz_valid).any(), f"Fold {numero} genera nulos en validación."
    assert matriz_train.shape[1] == matriz_valid.shape[1], f"Fold {numero} cambia de columnas."

    vistas = set(X.j.iloc[train_idx])
    diagnostico.append(
        {
            "dias_train": development[DATE].iloc[train_idx].dt.normalize().nunique(),
            "filas_train": len(train_idx),
            "j_en_train": len(vistas),
            "fin_train": fin_train.date(),
            "inicio_valid": inicio_valid.date(),
            "j_nuevas_pct": 100 * (~X.j.iloc[valid_idx].isin(vistas)).mean(),
            "fraude_valid_pct": 100 * y.iloc[valid_idx].mean(),
        }
    )

display(pd.DataFrame(diagnostico, index=[f"fold {i}" for i in range(len(folds))]).round(2))
assert development.monto.equals(load_periods()[0].monto)
print("Verificado: determinismo, orden temporal, sin nulos y monto original intacto.")

,dias_train,filas_train,j_en_train,fin_train,inicio_valid,j_nuevas_pct,fraude_valid_pct
fold 0,8,27361,4871,2020-03-15,2020-03-16,7.34,4.56
fold 1,15,52560,6236,2020-03-22,2020-03-23,3.81,5.69
fold 2,22,72629,6875,2020-03-29,2020-03-30,2.54,5.78
fold 3,29,90063,7269,2020-04-05,2020-04-06,2.31,5.41


Verificado: determinismo, orden temporal, sin nulos y monto original intacto.


## 3. Qué queda verificado y qué queda abierto

Los cuatro chequeos pasan: **la traducción no muestra fuga de información del futuro**. Por eso lo que se
guarda es el **Pipeline entrenado**, no una tabla de features: el Pipeline recalcula todo con el
entrenamiento de cada fold, y en producción aplica exactamente la misma transformación. Una tabla
calculada una sola vez sobre todo desarrollo filtraría la etiqueta sin que nada falle.

La tabla del último chequeo muestra algo a tener presente en el modelado: entre el **2,31% y el 7,34%** de
las filas de validación tienen un `j` que nunca apareció en entrenamiento, y el fold 0 aprende sobre
**4.871 categorías** contra las 7.269 del fold 3. Es la condición de producción, donde todos los días
entran categorías nuevas.

El 0,8467 con etiquetas reales es un **AUC de control, no la vara del problema**. La vara está en dinero:
superar los **+23.999** que rinde la mejor regla de un corte sobre `score` medida fuera de muestra, en
`03_modeling.ipynb`.